# Set up


In [ ]:
# package set up
!pip install -U openai pandas openpyxl pydantic

# google drive set up
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# API Set up

In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get("GPTKEY")
print(bool(OPENAI_API_KEY))

True


# Knownledge set up

In [ ]:
import os
import json
import time
import argparse
from typing import Dict, Any, List

from openai import OpenAI


def load_json(path: str) -> Dict[str, Any]:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_json(path: str, data: Dict[str, Any]) -> None:
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def normalize_files_cache(raw: Any) -> Dict[str, str]:
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}

    # Backward compatibility for old cache shapes like
    # [{"filename": "a.pdf", "file_id": "file-..."}, ...]
    if isinstance(raw, list):
        out: Dict[str, str] = {}
        for item in raw:
            if isinstance(item, dict) and item.get("filename") and item.get("file_id"):
                out[str(item["filename"])] = str(item["file_id"])
        return out

    return {}


def wait_for_vector_store_completion(client: OpenAI, vector_store_id: str, timeout_sec: int = 900, poll_sec: int = 5):
    start = time.time()
    last_status = None

    while True:
        vs = client.vector_stores.retrieve(vector_store_id)
        last_status = getattr(vs, "status", None)
        file_counts = getattr(vs, "file_counts", None)
        print(f"vector_store.status={last_status} file_counts={file_counts}")

        if last_status == "completed":
            return vs

        if last_status in {"expired", "failed", "cancelled"}:
            raise RuntimeError(f"Vector store entered terminal status: {last_status}")

        if time.time() - start > timeout_sec:
            raise TimeoutError(f"Timed out waiting for vector store completion. Last status: {last_status}")

        time.sleep(poll_sec)


def create_batch_and_wait(client: OpenAI, vector_store_id: str, file_ids: List[str]):
    # Prefer SDK helper when available.
    batches = getattr(client.vector_stores, "file_batches", None)
    if batches is None:
        raise RuntimeError("OpenAI SDK does not expose vector_stores.file_batches")

    if hasattr(batches, "create_and_poll"):
        batch = batches.create_and_poll(vector_store_id=vector_store_id, file_ids=file_ids)
        print(f"file_batch.status={getattr(batch, 'status', None)} file_counts={getattr(batch, 'file_counts', None)}")
        status = getattr(batch, "status", None)
        if status and status != "completed":
            raise RuntimeError(f"File batch did not complete successfully: {status}")
        return batch

    batch = batches.create(vector_store_id=vector_store_id, file_ids=file_ids)
    print(f"Created file batch: {getattr(batch, 'id', None)}")
    wait_for_vector_store_completion(client, vector_store_id)
    return batch


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--api_key", required=True)
    parser.add_argument("--knowledge_files", nargs="+", required=True)
    parser.add_argument("--vector_store_name", required=True)
    parser.add_argument("--cache_path", required=True)
    parser.add_argument("--force_new_vector_store", action="store_true")
    args = parser.parse_args()

    client = OpenAI(api_key=args.api_key)

    cache = load_json(args.cache_path)
    files_cache = normalize_files_cache(cache.get("files", {}))
    vector_store_id = None if args.force_new_vector_store else cache.get("vector_store", {}).get("id")

    uploaded_file_ids: List[str] = []

    for filepath in args.knowledge_files:
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File not found: {filepath}")

        filename = os.path.basename(filepath)

        if filename in files_cache:
            file_id = files_cache[filename]
            print(f"Reuse file: {filename} -> {file_id}")
        else:
            print(f"Uploading: {filename}")
            with open(filepath, "rb") as f:
                uploaded = client.files.create(file=f, purpose="assistants")
            file_id = uploaded.id
            files_cache[filename] = file_id
            print(f"Uploaded: {filename} -> {file_id}")

        uploaded_file_ids.append(file_id)

    if vector_store_id:
        print(f"Reuse vector store: {vector_store_id}")
    else:
        print("Creating vector store...")
        vs = client.vector_stores.create(name=args.vector_store_name)
        vector_store_id = vs.id
        print(f"Created vector store: {vector_store_id}")

    print("Attaching files to vector store and waiting for indexing...")
    create_batch_and_wait(client, vector_store_id, uploaded_file_ids)

    # Final readiness check.
    wait_for_vector_store_completion(client, vector_store_id)

    cache_out = {
        "files": files_cache,
        "vector_store": {"id": vector_store_id},
    }
    save_json(args.cache_path, cache_out)

    print("\nDONE")
    print("vector_store_id:", vector_store_id)
    print("cache_path:", args.cache_path)


if __name__ == "__main__":
    main()


usage: colab_kernel_launcher.py [-h] --api_key API_KEY --knowledge_files
                                KNOWLEDGE_FILES [KNOWLEDGE_FILES ...]
                                --vector_store_name VECTOR_STORE_NAME
                                --cache_path CACHE_PATH
                                [--force_new_vector_store]
colab_kernel_launcher.py: error: the following arguments are required: --api_key, --knowledge_files, --vector_store_name, --cache_path
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --api_key, --knowledge_files, --vector_store_name, --cache_path

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_31236/3710549459.py", line 145, in <cell line: 0>
    main()
  File "/tmp/ipykernel_31236/3710549459.py", line 90, in main
    args = parser.parse_args()
           ^^^^^^^^^^^

TypeError: object of type 'NoneType' has no len()

In [ ]:
!python build_knowledge.py \
  --api_key "$OPENAI_API_KEY" \
  --knowledge_files \
    "Medication errors definitions.pdf" \
    "taxonomy2001-07-31.pdf" \
    "BKN_Med_List (1).xlsx" \
  --vector_store_name "MediCheck Medication Error KB" \
  --cache_path "/content/drive/MyDrive/medicheck_cache/knowledge_cache.json"

Reuse file: Medication errors definitions.pdf -> file-B7xmjr2KFXfnGZQFQyybCJ
Reuse file: taxonomy2001-07-31.pdf -> file-43HbHcxNiUZELHLUB4BDXy
Reuse file: BKN_Med_List (1).xlsx -> file-7Jwdg7Q8xf96JgnNXckduh
Reuse vector store: vs_69bc2c678ed881919300e9a501670c0f
Attaching files to vector store and waiting for indexing...
file_batch.status=completed file_counts=FileCounts(cancelled=0, completed=3, failed=0, in_progress=0, total=3)
vector_store.status=completed file_counts=FileCounts(cancelled=0, completed=3, failed=0, in_progress=0, total=3)

DONE
vector_store_id: vs_69bc2c678ed881919300e9a501670c0f
cache_path: /content/drive/MyDrive/medicheck_cache/knowledge_cache.json


In [ ]:
!cat /content/drive/MyDrive/medicheck_cache/knowledge_cache.json

{
  "files": {
    "Medication errors definitions.pdf": "file-B7xmjr2KFXfnGZQFQyybCJ",
    "taxonomy2001-07-31.pdf": "file-43HbHcxNiUZELHLUB4BDXy",
    "BKN_Med_List (1).xlsx": "file-7Jwdg7Q8xf96JgnNXckduh"
  },
  "vector_store": {
    "id": "vs_69bc2c678ed881919300e9a501670c0f"
  }
}

In [ ]:
!rm -f /content/drive/MyDrive/medicheck_cache/knowledge_cache.json




# create run_inference.py


In [ ]:
import os
import json
import time
import argparse
import re
import traceback
from typing import List, Dict, Any

import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI, APIConnectionError, APITimeoutError, RateLimitError, APIError


class MediCheckNCCMERPOutput(BaseModel):
    order_id: str = Field(description="Prescription order identifier")
    summary: str = Field(description="Clinical summary of the prescription review")
    has_medication_error: bool = Field(description="True if a medication error is detected, otherwise False")
    error_categories: List[str] = Field(description="Multi-label medication error categories detected from the knowledge base. Use [] if none.")
    error_details: List[str] = Field(description="Detailed explanation for each detected medication error. Use [] if none.")
    overall_recommendation: str = Field(description="Overall recommendation for the prescription")
    ncc_merp_severity_category: str = Field(description="NCC MERP severity category: A, B, C, D, E, F, G, H, or I")
    ncc_merp_error_types: List[str] = Field(description="NCC MERP error type labels")
    ncc_merp_causes: List[str] = Field(description="Likely NCC MERP cause labels")
    medication_error_stage: List[str] = Field(description="Stage(s) in the medication-use process")
    prescribing_issue_type: List[str] = Field(description="prescribing fault, prescription error, or both")
    implicated_drugs: List[str] = Field(description="Drug names involved in the detected issue(s)")
    evidence_quotes: List[str] = Field(description="Short evidence statements")


JSON_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "order_id": {"type": "string"},
        "summary": {"type": "string"},
        "has_medication_error": {"type": "boolean"},
        "error_categories": {"type": "array", "items": {"type": "string"}},
        "error_details": {"type": "array", "items": {"type": "string"}},
        "overall_recommendation": {"type": "string"},
        "ncc_merp_severity_category": {
            "type": "string",
            "enum": ["A", "B", "C", "D", "E", "F", "G", "H", "I"],
        },
        "ncc_merp_error_types": {"type": "array", "items": {"type": "string"}},
        "ncc_merp_causes": {"type": "array", "items": {"type": "string"}},
        "medication_error_stage": {"type": "array", "items": {"type": "string"}},
        "prescribing_issue_type": {"type": "array", "items": {"type": "string"}},
        "implicated_drugs": {"type": "array", "items": {"type": "string"}},
        "evidence_quotes": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "order_id",
        "summary",
        "has_medication_error",
        "error_categories",
        "error_details",
        "overall_recommendation",
        "ncc_merp_severity_category",
        "ncc_merp_error_types",
        "ncc_merp_causes",
        "medication_error_stage",
        "prescribing_issue_type",
        "implicated_drugs",
        "evidence_quotes",
    ],
}


def load_json(path: str, default):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default


def save_json(path: str, data) -> None:
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def normalize_order_id(value: Any) -> str:
    if value is None:
        return "UNKNOWN"
    return str(value).strip()


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(how="all")
    df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
    df = df.where(pd.notnull(df), None)
    df = df.drop_duplicates()
    return df


def safe_read_excel(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Excel file not found: {path}")
    return pd.read_excel(path)


def dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    result = []
    for item in items or []:
        if item is None:
            continue
        item = str(item).strip()
        if not item:
            continue
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result


def get_instructions() -> str:
    return """
You are MediCheck, a clinical AI assistant for medication error detection.

Core scope:
- Review one prescription order at a time.
- Detect medication errors using MULTI-LABEL classification.
- Use ONLY the uploaded knowledge base available through file_search.
- Do NOT use external clinical knowledge.
- Ground your judgment in the uploaded medication error definition and taxonomy.
- Use the local medication list only as a supporting reference when relevant.

Definition rules:
- A medication error is a failure in the treatment process that leads to, or has the potential to lead to, harm to the patient.
- Distinguish prescribing fault from prescription error when possible.
- If the case does not support a medication error, mark has_medication_error as false.

NCC MERP rules:
- Assign one severity category only: A, B, C, D, E, F, G, H, or I.
- If no error is present but there is capacity to cause error, use A.
- If an error occurred but no harm is evident, consider B/C/D based on the available prescription-level evidence.
- Do not overstate harm when outcome data are unavailable.
- When outcome/harm is not provided in the prescription data, prefer the most conservative category justified by the evidence.

Error type rules:
- Use NCC MERP type labels when applicable, such as:
  Dose Omission,
  Improper Dose,
  Wrong Strength/Concentration,
  Wrong Drug,
  Wrong Dosage Form,
  Wrong Technique,
  Wrong Route of Administration,
  Wrong Rate,
  Wrong Duration,
  Wrong Time,
  Wrong Patient,
  Monitoring Error,
  Deteriorated Drug Error,
  Other

Cause rules:
- Use NCC MERP cause labels only when reasonably supported by the prescription data and retrieved knowledge.

Stage rules:
- Choose one or more from:
  prescribing,
  transcribing,
  dispensing,
  administration,
  monitoring

Prescribing classification rules:
- Use:
  prescribing fault
  prescription error
  both
- Use [] if not applicable.

Output rules:
- Return valid JSON that matches the required schema exactly.
- Do not add extra keys.
- Return JSON only. No markdown fences. No explanation outside JSON.
- If no medication error:
  - has_medication_error = false
  - error_categories = []
  - error_details = []
  - ncc_merp_error_types = []
  - implicated_drugs = []
- evidence_quotes must be short, factual, and based on the prescription data and retrieved knowledge.
"""


def build_user_prompt(order_id: str, prescription_records: List[Dict[str, Any]]) -> str:
    prescription_text = json.dumps(prescription_records, ensure_ascii=False, indent=2)
    return f"""
Review the following prescription order.

Order ID: {order_id}

Prescription data:
{prescription_text}

Tasks:
1. Decide whether this prescription contains a medication error.
2. Apply multi-label classification if more than one issue is present.
3. Classify the issue using the uploaded medication error definitions.
4. Map the issue to NCC MERP error type(s).
5. Assign one NCC MERP severity category A-I using conservative reasoning.
6. Identify the stage(s) involved in the medication-use process.
7. Distinguish prescribing fault versus prescription error when possible.
8. Provide concise evidence statements grounded in the prescription data and retrieved knowledge.
9. Give an overall recommendation.

Important:
- Use ONLY the uploaded knowledge base through file_search.
- Do NOT use external clinical knowledge.
- If the retrieved knowledge is insufficient, still return valid JSON using the most conservative interpretation and explain uncertainty in summary/error_details.
- Return JSON only.
"""


def extract_text_from_response(response: Any) -> str:
    output_text = getattr(response, "output_text", None)
    if isinstance(output_text, str) and output_text.strip():
        return output_text.strip()

    chunks = []
    output_items = getattr(response, "output", None) or []
    for item in output_items:
        if getattr(item, "type", None) == "message":
            for c in getattr(item, "content", None) or []:
                c_type = getattr(c, "type", None)
                if c_type in ("output_text", "text"):
                    text_value = getattr(c, "text", None)
                    if isinstance(text_value, str) and text_value.strip():
                        chunks.append(text_value.strip())
                elif c_type == "refusal":
                    refusal_text = getattr(c, "refusal", None)
                    if isinstance(refusal_text, str) and refusal_text.strip():
                        chunks.append(refusal_text.strip())

    joined = "\n".join(chunks).strip()
    if joined:
        return joined

    preview = response.model_dump() if hasattr(response, "model_dump") else str(response)
    raise ValueError(f"Model returned empty text output. Response preview: {str(preview)[:3000]}")


def extract_json_from_text(text: str) -> Dict[str, Any]:
    if text is None:
        raise ValueError("response text is None")
    text = text.strip()
    if not text:
        raise ValueError("response text is empty")

    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        return json.loads(match.group(0))

    raise ValueError(f"Could not parse valid JSON from model output: {repr(text[:1000])}")


def call_model(
    client: OpenAI,
    model: str,
    vector_store_id: str,
    instructions: str,
    user_prompt: str,
    max_num_results: int = 8,
    debug: bool = False,
    max_retries: int = 3,
) -> Dict[str, Any]:
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.responses.create(
                model=model,
                instructions=instructions,
                input=user_prompt,
                tools=[
                    {
                        "type": "file_search",
                        "vector_store_ids": [vector_store_id],
                        "max_num_results": max_num_results,
                    }
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "medicheck_ncc_merp_output",
                        "strict": True,
                        "schema": JSON_SCHEMA,
                    }
                },
            )

            if debug and hasattr(response, "model_dump"):
                preview = response.model_dump()
                print("----- RESPONSE OBJECT PREVIEW -----")
                print(str(preview)[:3000])
                print("----- END RESPONSE OBJECT PREVIEW -----")

            raw_text = extract_text_from_response(response)

            if debug:
                print("----- RAW MODEL TEXT START -----")
                print(repr(raw_text[:2000]))
                print("----- RAW MODEL TEXT END -----")

            return extract_json_from_text(raw_text)

        except (APIConnectionError, APITimeoutError, RateLimitError, APIError, ConnectionError) as e:
            last_error = e
            wait_sec = min(2 ** attempt, 10)
            print(f"⚠️ API/network error on attempt {attempt}/{max_retries}: {e}")
            time.sleep(wait_sec)
        except Exception as e:
            last_error = e
            wait_sec = min(2 ** attempt, 10)
            print(f"⚠️ Parse/empty-output error on attempt {attempt}/{max_retries}: {e}")
            time.sleep(wait_sec)

    raise last_error


def postprocess_output(parsed: Dict[str, Any], order_id: str) -> Dict[str, Any]:
    parsed["order_id"] = normalize_order_id(parsed.get("order_id", order_id))

    for key in [
        "error_categories",
        "error_details",
        "ncc_merp_error_types",
        "ncc_merp_causes",
        "medication_error_stage",
        "prescribing_issue_type",
        "implicated_drugs",
        "evidence_quotes",
    ]:
        parsed[key] = dedupe_preserve_order(parsed.get(key, []))

    if not parsed.get("has_medication_error", False):
        parsed["error_categories"] = []
        parsed["error_details"] = []
        parsed["ncc_merp_error_types"] = []
        parsed["implicated_drugs"] = []
        severity = parsed.get("ncc_merp_severity_category")
        if severity not in {"A", "B", "C", "D"}:
            parsed["ncc_merp_severity_category"] = "A"

    validated = MediCheckNCCMERPOutput(**parsed)
    return validated.model_dump()


def append_debug_log(path: str, payload: Dict[str, Any]) -> None:
    logs = load_json(path, [])
    logs.append(payload)
    save_json(path, logs)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--api_key", required=True)
    parser.add_argument("--excel_file", required=True)
    parser.add_argument("--group_column", required=True)
    parser.add_argument("--cache_path", default="/content/drive/MyDrive/medicheck_cache/knowledge_cache.json")
    parser.add_argument("--output_dir", default="/content/drive/MyDrive/medicheck_output")
    parser.add_argument("--model", default="gpt-4o-mini")
    parser.add_argument("--sleep_sec", type=float, default=1.0)
    parser.add_argument("--autosave_every", type=int, default=20)
    parser.add_argument("--max_num_results", type=int, default=8)
    parser.add_argument("--debug", action="store_true")
    parser.add_argument("--max_retries", type=int, default=3)
    args = parser.parse_args()

    client = OpenAI(api_key=args.api_key)

    cache = load_json(args.cache_path, {})
    vector_store_id = cache.get("vector_store", {}).get("id")
    if not vector_store_id:
        raise ValueError("vector_store_id not found in cache. Run build_knowledge_fixed.py first.")

    os.makedirs(args.output_dir, exist_ok=True)

    partial_file = os.path.join(args.output_dir, "partial_results.json")
    failed_file = os.path.join(args.output_dir, "failed_cases.json")
    final_file = os.path.join(args.output_dir, "medicheck_results.json")
    debug_file = os.path.join(args.output_dir, "debug_log.json")

    results: Dict[str, Any] = load_json(partial_file, {})
    failed_cases: List[Dict[str, str]] = load_json(failed_file, [])

    df = safe_read_excel(args.excel_file)
    df = clean_dataframe(df)

    if args.group_column not in df.columns:
        raise ValueError(f"Column '{args.group_column}' not found. Available columns: {list(df.columns)}")

    grouped = list(df.groupby(args.group_column, dropna=False))
    total_cases = len(grouped)

    remaining = []
    already_processed = 0
    for order_id, group_df in grouped:
        oid = normalize_order_id(order_id)
        if oid in results:
            already_processed += 1
        else:
            remaining.append((oid, group_df))

    print(f"Already processed: {already_processed}")
    print(f"Remaining: {len(remaining)} / {total_cases}")

    instructions = get_instructions()

    for idx, (order_id, group_df) in enumerate(remaining, start=1):
        current_case = already_processed + idx
        print(f"Processing {current_case}/{total_cases} -> Order {order_id}")

        raw_prompt_preview = None
        try:
            prescription_records = group_df.to_dict(orient="records")
            user_prompt = build_user_prompt(order_id, prescription_records)
            raw_prompt_preview = user_prompt[:3000]

            parsed = call_model(
                client=client,
                model=args.model,
                vector_store_id=vector_store_id,
                instructions=instructions,
                user_prompt=user_prompt,
                max_num_results=args.max_num_results,
                debug=args.debug,
                max_retries=args.max_retries,
            )

            final_output = postprocess_output(parsed, order_id)
            results[order_id] = final_output

        except ValidationError as e:
            print(f"❌ Validation error on {order_id}: {e}")
            failed_cases.append({
                "order_id": order_id,
                "error_type": "ValidationError",
                "error": str(e),
                "traceback": traceback.format_exc(),
            })
            append_debug_log(debug_file, {
                "order_id": order_id,
                "error_type": "ValidationError",
                "error": str(e),
                "traceback": traceback.format_exc(),
                "prompt_preview": raw_prompt_preview,
            })
        except Exception as e:
            print(f"❌ Error on {order_id}: {e}")
            failed_cases.append({
                "order_id": order_id,
                "error_type": type(e).__name__,
                "error": str(e),
                "traceback": traceback.format_exc(),
            })
            append_debug_log(debug_file, {
                "order_id": order_id,
                "error_type": type(e).__name__,
                "error": str(e),
                "traceback": traceback.format_exc(),
                "prompt_preview": raw_prompt_preview,
            })

        time.sleep(args.sleep_sec)

        if current_case % args.autosave_every == 0:
            save_json(partial_file, results)
            save_json(failed_file, failed_cases)
            print(f"Autosaved after {current_case} cases")

    save_json(final_file, results)
    save_json(failed_file, failed_cases)

    print("\nDONE")
    print(f"Total processed: {len(results)}")
    print(f"Failed: {len(failed_cases)}")
    print(f"Results saved to: {final_file}")
    print(f"Debug log saved to: {debug_file}")


if __name__ == "__main__":
    main()


# อัปโหลด knowledge files 3 ไฟล์เข้า Colab

In [ ]:
from google.colab import files
uploaded = files.upload()

# Upload + Clean data

In [ ]:
!pip -q install openai pandas xlrd

import os
import json
import time
import pandas as pd
from openai import OpenAI
from google.colab import files, userdata

# ==============================
# 1) LOAD FILE
# ==============================
# Check if file_name is already defined (from a previous run)
if 'file_name' not in globals():
    uploaded = files.upload()
    file_name = next(iter(uploaded))
else:
    print(f"Using previously uploaded file: {file_name}")

df = pd.read_excel(file_name)

# ==============================
# 2) CLEAN DATA
# ==============================
df = df.dropna(how="all")
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df = df.where(pd.notnull(df), None)
df = df.drop_duplicates()

# Remove sensitive
SENSITIVE_COLUMNS = ["Careprovider", "HN", "EN", "Order Date",
                   "Order Start", "Order End", "Order Type", "Category Type",
                   "Category", "Sub Category", "Drug Form", "Drug Instruction",
                   "Instruction", "Doctor Instruction", "Dispensing Instruction",
                   "Patient Instruction", "Ward", "Bed", "Ordering User",
                   "Department", "Department.1", "Order To", "Department.2",
                   "POM", "Order Type.1", "Parent"] # 'Orderl Number' removed from this list
df = df.drop(columns=[c for c in SENSITIVE_COLUMNS if c in df.columns], errors="ignore")

# ==============================
# 3) GROUP DATA
# ==============================
GROUP_COL = "Orderl Number"

if GROUP_COL not in df.columns:
    raise ValueError(f"❌ Column '{GROUP_COL}' not found")

grouped = df.groupby(GROUP_COL)

# อัปโหลดไฟล์ prescription Excel

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving run_inference_fixed.py to run_inference_fixed.py
Saving build_knowledge_fixed.py to build_knowledge_fixed.py


# Clean data + Blind ID

In [ ]:
import pandas as pd
import hashlib
import os

# ========= 1) ตั้งชื่อไฟล์ต้นฉบับ =========
input_file = "RT_COMMON_904_test.xlsx"

# ========= 2) อ่านไฟล์ =========
df = pd.read_excel(input_file)

# ========= 3) เลือกเฉพาะคอลัมน์ที่ต้องใช้ =========
required_cols = ["Orderl Number", "Generic Name", "Dosage", "Dosage UOM", "Frequency"]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

work_df = df[required_cols].copy()

# ========= 4) ลบแถวว่าง =========
work_df = work_df.dropna(how="all")

# ========= 5) strip space =========
for col in work_df.columns:
    if work_df[col].dtype == "object":
        work_df[col] = work_df[col].astype(str).str.strip()

# แทนค่า nan string กลับเป็น None
work_df = work_df.replace({"nan": None, "None": None, "": None})

# ========= 6) สร้าง blinded ID จาก Orderl Number =========
# ใส่ salt เพื่อให้เดาย้อนกลับยากขึ้น
SALT = "medicheck_blinding_v1"

def encrypt_order_id(order_no):
    if pd.isna(order_no) or order_no is None:
        return None
    raw = f"{SALT}_{str(order_no).strip()}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:12]

work_df["ID"] = work_df["Orderl Number"].apply(encrypt_order_id)

# ========= 7) สร้าง Dose =========
def build_dose(row):
    dose = row["Dosage"]
    dose_uom = row["Dosage UOM"]

    if dose is None and dose_uom is None:
        return None
    if dose is None:
        return str(dose_uom)
    if dose_uom is None:
        return str(dose)

    return f"{dose} {dose_uom}"

work_df["Drug"] = work_df["Generic Name"]
work_df["Dose"] = work_df.apply(build_dose, axis=1)
work_df["Frequency"] = work_df["Frequency"]

# ========= 8) เลือกเฉพาะ output format =========
clean_df = work_df[["ID", "Drug", "Dose", "Frequency"]].copy()

# ========= 9) ลบ duplicates =========
clean_df = clean_df.drop_duplicates()

# ========= 10) preview =========
print("Cleaned shape:", clean_df.shape)
display(clean_df.head(10))

# ========= 11) save =========
output_file = "RT_COMMON_904_test_clean_blinded.xlsx"
clean_df.to_excel(output_file, index=False)

print(f"Saved to: {output_file}")

# เช็คชื่อ column ก่อน

In [ ]:
import pandas as pd

df = pd.read_excel("RT_COMMON_904_test_clean_blinded.xlsx")
print(df.columns.tolist())
df.head()

['ID', 'Drug', 'Dose', 'Frequency']


,ID,Drug,Dose,Frequency
0,67ec75e67ef0,diazepam,10 Milligram,Once
1,67ec75e67ef0,"ondansetron, ondansetron hydrochloride",8 Milligram,Once
2,67ec75e67ef0,"hyoscine-n-butyl bromide, hyoscine-n-",20 Milligram,Take Now
3,0280d683793d,diazepam,1 Capsule,2 Time Daily Morning and Evening
4,0280d683793d,ondansetron,1 Tablet,3 Times Daily Before Breakfast Lunch and


# Run inference

In [ ]:
import traceback
from openai import APIConnectionError, APITimeoutError, RateLimitError, APIError

!python run_inference.py \
  --api_key "$OPENAI_API_KEY" \
  --excel_file "RT_COMMON_904_test_clean_blinded.xlsx" \
  --group_column "ID" \
  --cache_path "/content/drive/MyDrive/medicheck_cache/knowledge_cache.json" \
  --output_dir "/content/drive/MyDrive/medicheck_output" \
  --model "gpt-4o-mini"

Already processed: 0
Remaining: 56 / 56
Processing 1/56 -> Order 0280d683793d
⚠️ Parse/empty-output error on attempt 1/3: Model returned empty text output. Response preview: {'id': 'resp_002a3bde4b6fc4e80069bc3790ab4481958ae65522bac2be97', 'created_at': 1773942672.0, 'error': None, 'incomplete_details': None, 'instructions': '\nYou are MediCheck, a clinical AI assistant for medication error detection.\n\nCore scope:\n- Review one prescription order at a time.\n- Detect medication errors using MULTI-LABEL classification.\n- Use ONLY the uploaded knowledge base available through file_search.\n- Do NOT use external clinical knowledge.\n- Ground your judgment in the uploaded medication error definition and taxonomy.\n- Use the local medication list only as a supporting reference when relevant.\n\nDefinition rules:\n- A medication error is a failure in the treatment process that leads to, or has the potential to lead to, harm to the patient.\n- Distinguish prescribing fault from prescriptio

In [ ]:
!python run_inference.py \
  --api_key "$OPENAI_API_KEY" \
  --excel_file "RT_COMMON_904_test_clean_blinded.xlsx" \
  --group_column "ID" \
  --cache_path "/content/drive/MyDrive/medicheck_cache/knowledge_cache.json" \
  --output_dir "/content/drive/MyDrive/medicheck_output" \
  --model "gpt-4o-mini" \
  --debug

Already processed: 0
Remaining: 56 / 56
Processing 1/56 -> Order 0280d683793d
----- RESPONSE OBJECT PREVIEW -----
{'id': 'resp_0ed487414699fa2d0069bc37eb1aac81959df9a3fdecb62240', 'created_at': 1773942763.0, 'error': None, 'incomplete_details': None, 'instructions': '\nYou are MediCheck, a clinical AI assistant for medication error detection.\n\nCore scope:\n- Review one prescription order at a time.\n- Detect medication errors using MULTI-LABEL classification.\n- Use ONLY the uploaded knowledge base available through file_search.\n- Do NOT use external clinical knowledge.\n- Ground your judgment in the uploaded medication error definition and taxonomy.\n- Use the local medication list only as a supporting reference when relevant.\n\nDefinition rules:\n- A medication error is a failure in the treatment process that leads to, or has the potential to lead to, harm to the patient.\n- Distinguish prescribing fault from prescription error when possible.\n- If the case does not support a med

# check model

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

models = client.models.list()

for m in models.data:
    print(m.id)

gpt-4-0613
gpt-4
gpt-3.5-turbo
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
dall-e-3
dall-e-2
gpt-4-1106-preview
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-4-0125-preview
gpt-4-turbo-preview
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
gpt-4o-audio-preview
gpt-4o-realtime-preview
omni-moderation-latest
omni-moderation-2024-09-26
gpt-4o-realtime-preview-2024-12-17
gpt-4o-audio-preview-2024-12-17
gpt-4o-mini-realtime-preview-2024-12-17
gpt-4o-mini-audio-preview-2024-12-17
gpt-4o-mini-realtime-preview
gpt-4o-mini-audio-preview
gpt-4o-2024-11-20
gpt-3.5-turbo-16k
tts-1
whisper-1
text-embedding-ada-002


In [ ]:
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

resp = client.responses.create(
    model="gpt-4o-mini",
    input="Return JSON: {\"ok\": true}",
    text={"format": {"type": "json_object"}}
)

print(resp.output_text)

{"ok": true}
